In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("./data/santander_train.csv", encoding='latin-1')
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print(df['TARGET'].value_counts())


In [ ]:
unsatisfied_cnt = df[df['TARGET'] == 1].TARGET.count()
total_cnt = df.TARGET.count()
print(unsatisfied_cnt, total_cnt)
print('unsatisfied 비율=',(unsatisfied_cnt / total_cnt))

In [ ]:
df.describe()

In [ ]:
df['var3'].value_counts()

In [ ]:
df['var3'] = df['var3'].replace(-999999, 2)

# ID 또는 ID_code가 실제로 있을 때만 삭제
df.drop(columns=['ID', 'ID_code'], errors='ignore', inplace=True)
df.head()

In [ ]:
X_features = df.iloc[:, :-1]
y_labels = df.iloc[:, -1]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels, test_size=0.2, random_state=42)

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

xgb_clf = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    random_state=42,
    early_stopping_rounds=50,
    eval_metric='auc'
)

xgb_clf.fit(
    X_tr,
    y_tr,
    eval_set=[
        (X_tr, y_tr),
        (X_val, y_val)
    ],
    verbose=True
);

In [ ]:
xgb_roc_score = roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:, 1])
print(xgb_roc_score)

In [ ]:
from hyperopt import hp

xgb_search_space = {
    'max_depth': hp.quniform('max_depth', 5, 15, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 6, 1),
    'colsample_bytree': hp.quniform('colsample_bytree',0.5,0.95,0.05),
    'learning_rate' : hp.quniform('learning_rate', 0.01, 0.2, 0.01)
}

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import numpy as np

def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators=400
                            ,max_depth=int(search_space['max_depth']),
                            min_child_weight = int(search_space['min_child_weight']),
                            colsample_bytree= search_space['colsample_bytree'],
                            learning_rate = search_space['learning_rate'],
                            early_stopping_rounds=30,
                            eval_metric = 'auc')

    roc_auc_lists = []

    kf = KFold(n_splits=3)

    for tr_index, val_index in kf.split(X_train):
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]

        xgb_clf.fit(X_tr, y_tr,  eval_set = [(X_tr, y_tr),(X_val,y_val)]
                    )
        score = roc_auc_score(y_val, xgb_clf.predict_proba(X_val)[:,1])
        roc_auc_lists.append(score)

    return -1 * np.mean(roc_auc_lists)
    

In [ ]:
from hyperopt import fmin,tpe,Trials

trials = Trials()

best = fmin(fn= objective_func, space= xgb_search_space,
            algo= tpe.suggest, max_evals=50,
            trials= trials, rstate = np.random.default_rng(seed=30))

print(best)

In [ ]:
xgb_clf=XGBClassifier(n_estimators=400,
                      learning_rate=round(best['learning_rate'],5),
                      max_depth=int(best['min_child_weight']),
                      colsample_bytree=round(best['colsample_bytree'],5),
                      early_stopping_rounds=30, eval_metric = 'auc')




xgb_clf.fit(X_tr, y_tr,  eval_set = [(X_tr, y_tr),(X_val,y_val)])

xgb_roc_auc=roc_auc_score(y_test, xgb_clf.predict_proba(X_test)[:,1])
print(xgb_roc_auc)

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1,1,figsize=(10,8))
plot_importance(xgb_clf, ax=ax, max_num_features=20, height=0.4)

In [ ]:
from lightgbm import LGBMClassifier
lgbm_clf=LGBMClassifier(n_estimators=500)
eval_set=[(X_tr,y_tr),(X_val,y_val)]
lgbm_clf.fit(X_tr,y_tr,eval_metric='auc',eval_set=eval_set)
lgbm_roc_auc=roc_auc_score(y_test,lgbm_clf.predict_proba(X_test)[:,1])
print(lgbm_roc_auc)

In [ ]:
lgbm_search_space={
    'num_leaves':hp.quniform('num_leaves',21,64,1),
    'max_depth':hp.quniform('max_depth',100,160,1),
    'min_child_samples':hp.quniform('min_child_samples',60,100,1),
    'subsample':hp.quniform('subsample',0.7,1,0.1),
    'learning_rate':hp.quniform('learning_rate',0.01,0.2,0.01)
}

In [ ]:
def lgbm_objective_func(search_space):
    lgbm_clf = LGBMClassifier(
        n_estimators = 100,
        max_depth = int(search_space["max_depth"]),
        num_leaves = int(search_space["num_leaves"]),
        min_child_samples = int(search_space["min_child_samples"]),
        subsample = search_space["subsample"],
        learning_rate = search_space["learning_rate"],
        early_stopping_rounds = 30, 
        eval_metric = "auc"
    )

    kf = KFold(n_splits = 3)

    roc_auc_lists = []

    for tr_index, val_index in kf.split(X_train):
        X_tr, y_tr = X_train.iloc[tr_index], y_train.iloc[tr_index]
        X_val, y_val = X_train.iloc[val_index], y_train.iloc[val_index]

        lgbm_clf.fit(X_tr, y_tr, eval_set = [(X_tr, y_tr), (X_val, y_val)])

        score = roc_auc_score(y_val, lgbm_clf.predict_proba(X_val)[:, 1])
        roc_auc_lists.append(score)

    return -1 * np.mean(roc_auc_lists)

In [ ]:
from hyperopt import fmin, tpe, Trials

trials = Trials()

best = fmin(
    fn = lgbm_objective_func,
    space = lgbm_search_space,
    algo = tpe.suggest,
    max_evals = 50,
    trials = trials,
    rstate = np.random.default_rng(seed = 30)
)

print(best)

In [ ]:
lgbm_clf=lgbm_clf = LGBMClassifier(
        n_estimators = 100,
        max_depth = int(best["max_depth"]),
        num_leaves = int(best["num_leaves"]),
        min_child_samples = int(best["min_child_samples"]),
        subsample = best["subsample"],
        learning_rate = best["learning_rate"],
        early_stopping_rounds = 100, 
        eval_metric = "auc"
    )

lgbm_clf.fit(X_tr, y_tr, eval_set=[(X_tr,y_tr),(X_val,y_val)])

lgbm_roc_auc=roc_auc_score(y_test, lgbm_clf.predict_proba(X_test)[:,1])
print(lgbm_roc_auc)